# 🚗⚡ The EV Buyers Mystery — a Detective Story for Data Kids (and Grown-ups)

Welcome, detective! 🕵️

In this competition we try to guess **who will buy an electric vehicle (EV)**. Every row is one pretend person: their age, income, city, how far they drive to work, how worried they are about running out of battery ("range anxiety"), and finally the answer we want: **Will_Buy_EV — Yes or No**.

But here is the fun part. The competition data was *inspired by* a small **original dataset of 10,000 pretend people**, and that original dataset was **invented by a computer program**. A program that rolled dice. 🎲

In this notebook we will:

1. 👀 Meet the people in the original dataset.
2. 🎲 Play detective and figure out **exactly which dice the computer rolled** to invent them — we will even find the computer's lucky number!
3. 🏙️ Discover the hidden **rules** that connect cities, charging stations and worry.
4. 🧮 Uncover the **secret recipe** that decides who buys an EV.
5. 🤖 Build two simple starter models for the competition and make a submission file.

No scary math is needed. If you can read a bar chart, you can follow along. Let's go!

## Chapter 0 — Getting our detective tools ready 🧰

We load a few Python libraries and find the data files. The notebook works both on Kaggle (files under `/kaggle/input/...`) and on a local computer where the files sit one folder up.

In [ ]:
import glob, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)

def find_file(name):
    for root in ["/kaggle/input", ".", ".."]:
        hits = glob.glob(f"{root}/**/{name}", recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(name)

ORIG = find_file("EV_Adoption_and_Range_Anxiety_Dataset.csv")
TRAIN, TEST, SUB = find_file("train.csv"), find_file("test.csv"), find_file("sample_submission.csv")
orig = pd.read_csv(ORIG); train = pd.read_csv(TRAIN); test = pd.read_csv(TEST)
print(f"original dataset: {orig.shape[0]:,} rows x {orig.shape[1]} columns")
print(f"competition train: {train.shape[0]:,} rows,   test: {test.shape[0]:,} rows")

## Chapter 1 — Meet the pretend people 👋

Let's look at five of them. Each column is one fact about a person.

In [ ]:
orig.head()

Here is what the columns mean, in plain words:

| column | what it is |
|---|---|
| `Age` | how old the person is |
| `Gender` | Male / Female / Other |
| `Annual_Income_USD` | money earned per year, in dollars |
| `City_Type` | Urban (big city), Suburban (edge of a city) or Rural (countryside) |
| `Number_of_Cars_Owned` | how many cars they already have |
| `Current_Car_Type` | Sedan, SUV, Hatchback or Truck |
| `Daily_Commute_km` | how far they drive to work every day |
| `Charging_Stations_Near_Home` / `_Near_Work` | how many EV chargers are close to home / work |
| `Home_Charging_Possible` | can they plug in at home? |
| `Environmental_Concern_Level` | 1 (don't care) to 5 (care a lot) about the planet 🌍 |
| `Subsidy_Available` | does the government give them money towards an EV? |
| `Range_Anxiety_Level` | Low / Medium / High worry about running out of battery 🔋 |
| `Will_Buy_EV` | **the answer we want to predict** |

A few cells are empty (`NaN`) — about 2% of the incomes, commutes and concern levels are missing in the original file. Detectives notice such things. 🔍

In [ ]:
missing = orig.isna().sum(); missing = missing[missing > 0]
print("missing values per column:"); print(missing.to_string())
print(f"\nBuyers: {(orig.Will_Buy_EV == 'Yes').mean():.1%} of the 10,000 people say Yes to an EV.")

### A first look at everyone at once 📊

Bar charts for the "category" columns (things with a few possible values) and histograms for the number columns.

In [ ]:
cat_cols = ["Gender", "City_Type", "Number_of_Cars_Owned", "Current_Car_Type", "Home_Charging_Possible", "Environmental_Concern_Level", "Subsidy_Available", "Range_Anxiety_Level", "Will_Buy_EV"]
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, c in zip(axes.ravel(), cat_cols):
    vc = orig[c].value_counts(dropna=False).sort_index()
    sns.barplot(x=vc.index.astype(str), y=vc.values, ax=ax, hue=vc.index.astype(str), legend=False)
    ax.set_title(c, fontweight="bold"); ax.set_xlabel(""); ax.set_ylabel("people")
    for i, v in enumerate(vc.values): ax.text(i, v, f"{v/len(orig):.0%}", ha="center", va="bottom", fontsize=9)
plt.suptitle("The original 10,000 people — who are they?", fontsize=16, fontweight="bold", y=1.01); plt.tight_layout(); plt.show()

In [ ]:
num_cols = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, c in zip(axes, num_cols):
    bins = 45 if c == "Age" else (20 if "Stations" in c else 60)
    ax.hist(orig[c].dropna(), bins=bins, color="#66c2a5", edgecolor="white"); ax.set_title(c, fontweight="bold"); ax.set_ylabel("people")
plt.suptitle("Number columns", fontsize=16, fontweight="bold", y=1.03); plt.tight_layout(); plt.show()

Look closely at the shapes. 👆

* **Age** is *flat*: every age from 25 to 69 has about the same number of people. That is the shape you get when you roll a fair die — a 45-sided die! 🎲
* **Income** looks like a **bell** 🔔 ... with a strange tall spike at the very left. Something odd happened at $30,000. Keep that in mind.
* **Commute** is also a bell, squished against a wall on the left at 5 km.
* **Charging stations** have odd stair shapes. We will solve that mystery in Chapter 3.

Time to become dice detectives.

## Chapter 2 — The dice detective 🎲

**Big idea:** a computer "invents" a person by rolling dice. A fair die gives a *flat* histogram. A "bell-curve die" (a normal distribution) gives a bell shape. If we can guess the dice, we can guess the program.

### 2.1 Age: a 45-sided die

If every age from 25 to 69 is equally likely, each age should appear about 10,000 / 45 = 222 times. Let's check.

In [ ]:
age_counts = orig.Age.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(age_counts.index, age_counts.values, color="#8da0cb", edgecolor="white")
ax.axhline(10000 / 45, color="red", ls="--", lw=2, label="expected 222 people per age if the die is fair")
ax.set_xlabel("Age"); ax.set_ylabel("people"); ax.set_title("Age: every number from 25 to 69 shows up about 222 times → a fair 45-sided die", fontweight="bold"); ax.legend(); plt.show()
print(f"smallest count {age_counts.min()}, largest {age_counts.max()}, ages present: {age_counts.index.min()} to {age_counts.index.max()}")

✅ Case closed: `Age = a random whole number from 25 to 69`.

### 2.2 Income: a bell curve with a floor

A normal distribution is what you get when many small random pushes add up — heights of people, for example. Its histogram is a bell. Our income bell has its top near **$85,000**, and it is about **$35,000** wide on each side.

But what about the spike on the left? Look at the smallest income in the data...

In [ ]:
inc = orig.Annual_Income_USD.dropna()
print(f"smallest income: ${inc.min():,.0f}   |  how many people have EXACTLY $30,000: {(inc == 30000).sum()} ({(inc == 30000).mean():.1%})")
print(f"mean income: ${inc.mean():,.0f}   standard deviation: ${inc.std():,.0f}")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].hist(inc, bins=80, color="#fc8d62", edgecolor="white"); axes[0].set_title("Real incomes: a bell\nwith a spike at $30,000", fontweight="bold", fontsize=13); axes[0].set_xlabel("income ($)")
rng = np.random.RandomState(0); fake = np.clip(rng.normal(85_000, 35_000, 10_000), 30_000, None)
axes[1].hist(fake, bins=80, color="#a6d854", edgecolor="white"); axes[1].set_title("Our guess: bell(85,000 ± 35,000),\nand anything below $30,000 → $30,000", fontweight="bold", fontsize=13); axes[1].set_xlabel("income ($)")
plt.tight_layout(); plt.show()

The two pictures match! 🎉 The program drew a bell-curve income and then said: *"nobody earns less than $30,000 — bump them up."* Those bumped people make the spike (about 6% of everyone in the original file).

### 2.3 Commute: the same trick with a 5 km floor

Bell centred at **40 km**, about **25 km** wide, and everything below 5 km becomes exactly 5.0 km. It is rounded to one decimal.

In [ ]:
km = orig.Daily_Commute_km.dropna()
print(f"smallest commute {km.min()} km; exactly 5.0 km: {(km == 5.0).mean():.1%} of people; mean {km.mean():.1f}, std {km.std():.1f}")
fig, ax = plt.subplots(figsize=(8, 4)); ax.hist(km, bins=80, color="#e78ac3", edgecolor="white")
ax.set_title("Commute: bell(40 ± 25 km) with a floor at 5 km", fontweight="bold"); ax.set_xlabel("km per day"); plt.show()

### 2.4 The category dice

For the "word" columns the computer used loaded dice: each face has its own chance. Look at the percentages on the bar charts of Chapter 1 and compare with these suspiciously round numbers:

| column | our guess for the dice |
|---|---|
| Gender | Male 52%, Female 45%, Other 3% |
| City_Type | Urban 50%, Suburban 35%, Rural 15% |
| Number_of_Cars_Owned | 1 → 40%, 2 → 40%, 3 → 15%, 4 → 5% |
| Current_Car_Type | Sedan 40%, SUV 35%, Hatchback 15%, Truck 10% |
| Environmental_Concern_Level | 1,2,3,4,5 each 20% (a fair 5-sided die) |
| Subsidy_Available | Yes 60%, No 40% |

Let's put the real shares next to the guesses.

In [ ]:
guesses = {"Gender": {"Male": .52, "Female": .45, "Other": .03}, "City_Type": {"Urban": .50, "Suburban": .35, "Rural": .15}, "Number_of_Cars_Owned": {1: .40, 2: .40, 3: .15, 4: .05},
           "Current_Car_Type": {"Sedan": .40, "SUV": .35, "Hatchback": .15, "Truck": .10}, "Environmental_Concern_Level": {k: .2 for k in [1., 2., 3., 4., 5.]}, "Subsidy_Available": {"Yes": .60, "No": .40}}
rows = []
for c, g in guesses.items():
    share = orig[c].value_counts(normalize=True)
    for k, p in g.items(): rows.append((c, str(k), share.get(k, 0), p))
cmp = pd.DataFrame(rows, columns=["column", "value", "real share", "guessed dice"])
cmp["real share"] = cmp["real share"].map("{:.1%}".format); cmp["guessed dice"] = cmp["guessed dice"].map("{:.0%}".format)
cmp

Every real share is within a whisker of a round number. When 10,000 dice rolls land so close to 52/45/3 or 40/40/15/5, you can be pretty sure those were the loaded dice.

### 2.5 The computer's lucky number 🍀

Here comes the coolest trick in the whole notebook.

Computers do not roll real dice. They use a **pseudo-random number generator**: a formula that produces a long list of numbers that *look* random. The list depends on a starting number called the **seed**. Same seed → same list, every time.

If the program used NumPy's classic generator, and we guess the seed and the *order* of the columns, we should be able to **re-create the original data exactly**, number for number. Let's try seed 101 with the dice we discovered, in the order the columns appear in the file.

In [ ]:
n = len(orig); rng = np.random.RandomState(101)
age    = rng.randint(25, 70, n)
gender = rng.choice(["Male", "Female", "Other"], n, p=[.52, .45, .03])
income = np.clip(rng.normal(85_000, 35_000, n), 30_000, None).astype(int)
city   = rng.choice(["Urban", "Suburban", "Rural"], n, p=[.50, .35, .15])
cars   = rng.choice([1, 2, 3, 4], n, p=[.40, .40, .15, .05])
ctype  = rng.choice(["Sedan", "SUV", "Hatchback", "Truck"], n, p=[.40, .35, .15, .10])
commute = np.clip(rng.normal(40, 25, n), 5, None).round(1)
ok_inc = orig.Annual_Income_USD.notna(); ok_km = orig.Daily_Commute_km.notna()
checks = {"Age": (age == orig.Age).mean(), "Gender": (gender == orig.Gender).mean(), "Income (where not missing)": (income[ok_inc] == orig.Annual_Income_USD[ok_inc]).mean(),
          "City": (city == orig.City_Type).mean(), "Cars": (cars == orig.Number_of_Cars_Owned).mean(), "Car type": (ctype == orig.Current_Car_Type).mean(), "Commute (where not missing)": np.isclose(commute[ok_km], orig.Daily_Commute_km[ok_km]).mean()}
for k, v in checks.items(): print(f"{k:28s} matches the real file: {v:.1%}")

🤯 **100% match, for all 10,000 people, in seven columns.** We did not fit anything — we simply rolled the same dice in the same order with the same seed (101), and the computer produced the very same people. This is the strongest possible proof of how the original dataset was made.

*(Fun fact: because we re-rolled the dice, we can even see the incomes that were later blanked out as missing values. The program invented them first and hid some afterwards.)*

## Chapter 3 — Cities, chargers and plugs 🏙️🔌

Remember the strange stair-shaped charger histograms? The trick is that the dice for chargers depend on the **city**. Big cities have many chargers, the countryside has few. Let's look at the chargers separately for each city type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, c in zip(axes, ["Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]):
    tab = pd.crosstab(orig[c], orig.City_Type)
    tab.plot(kind="bar", ax=ax, width=0.85, edgecolor="white"); ax.set_title(c, fontweight="bold"); ax.set_xlabel("number of chargers"); ax.set_ylabel("people"); ax.tick_params(axis="x", rotation=0)
plt.suptitle("Chargers depend on the city: each city rolls its own fair die over its own range", fontsize=15, fontweight="bold", y=1.03); plt.tight_layout(); plt.show()
rng_tab = orig.groupby("City_Type")[["Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]].agg(["min", "max"]); rng_tab

Each city has its own **fair die over a range**:

| city | chargers near home | chargers near work |
|---|---|---|
| Rural | 0 – 2 | 0 – 3 |
| Suburban | 0 – 7 | 1 – 9 |
| Urban | 2 – 14 | 3 – 19 |

Within a range every number is equally likely (the bars are flat inside each colour). Stack three flat shapes of different widths and you get the "stairs" we saw in Chapter 1. Mystery solved! 🪜

**Home charging** also depends on the city: it is much easier to plug in at a house in the countryside than in a city apartment.

In [ ]:
home_rate = orig.groupby("City_Type").Home_Charging_Possible.apply(lambda s: (s == "Yes").mean()).reindex(["Urban", "Suburban", "Rural"])
fig, ax = plt.subplots(figsize=(7, 4)); bars = ax.bar(home_rate.index, home_rate.values, color=["#8da0cb", "#66c2a5", "#a6d854"], edgecolor="white")
for b, v, g in zip(bars, home_rate.values, [.40, .80, .90]): ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.1%}\n(die: {g:.0%})", ha="center", fontweight="bold")
ax.set_ylim(0, 1.05); ax.set_ylabel("share with home charging"); ax.set_title("Can they charge at home? A coin flip whose fairness depends on the city", fontweight="bold"); plt.show()

Urban 40%, Suburban 80%, Rural 90% — three loaded coins, one per city. 🪙

We can now draw the whole **family tree** of the data so far: most columns are independent dice, but the chargers and the home plug are *children* of the city.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5)); ax.axis("off")
def box(x, y, text, color, w=2.6, h=0.9):
    ax.add_patch(patches.FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.1", fc=color, ec="black", lw=1.5)); ax.text(x, y, text, ha="center", va="center", fontsize=10.5, fontweight="bold")
def arrow(x1, y1, x2, y2): ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", lw=2, color="black"))
top = [("Age\n(die 25–69)", "#ffd92f"), ("Gender\n(loaded die)", "#ffd92f"), ("Income\n(bell, floor 30k)", "#ffd92f"), ("City\n(loaded die)", "#fc8d62"), ("Cars\n(loaded die)", "#ffd92f"), ("Car type\n(loaded die)", "#ffd92f"), ("Commute\n(bell, floor 5 km)", "#ffd92f"), ("Concern\n(die 1–5)", "#ffd92f"), ("Subsidy\n(60% coin)", "#ffd92f")]
xs = np.linspace(1.6, 22.4, len(top))
for (t, c), x in zip(top, xs): box(x, 5, t, c)
box(5.0, 2.8, "Chargers near home\n(die, range by city)", "#8da0cb", w=3.4); box(9.5, 2.8, "Chargers near work\n(die, range by city)", "#8da0cb", w=3.4); box(14.0, 2.8, "Home charging\n(coin by city)", "#8da0cb", w=3.4)
cx = xs[3]
for x in (5.0, 9.5, 14.0): arrow(cx, 4.55, x, 3.25)
box(10.2, 0.8, "Range anxiety  →  Will buy EV?   (Chapters 4 and 5)", "#e78ac3", w=10.5, h=0.9)
for x in (5.0, 9.5, 14.0): arrow(x, 2.35, 10.2, 1.25)
arrow(xs[6], 4.55, 14.6, 1.25)
ax.set_xlim(0, 24); ax.set_ylim(0, 6); ax.set_title("How the computer invents one person: roll the yellow dice, let the city decide the blue ones, then compute the pink ones", fontsize=13, fontweight="bold"); plt.show()

## Chapter 4 — The worry meter 😰🔋

`Range_Anxiety_Level` (Low / Medium / High) is not a die at all. It is **computed** from four things, plus a little randomness:

> **worry score = commute − 5 × (chargers near home) − 5 × (chargers near work) − 150 × (home charging) + random wobble**

* a long commute makes you worry more (+1 per km),
* every charger near home or work makes you worry less (−5 each),
* being able to charge at home makes you worry *much* less (−150),
* the wobble is a bell-curve nudge with a spread of about 50.

Then: score ≤ −25 → **Low**, between −25 and 75 → **Medium**, above 75 → **High**.

Let's compute the score without the wobble and see how the three worry levels sit on it.

In [ ]:
o = orig.copy(); o["Daily_Commute_km"] = o.Daily_Commute_km.fillna(pd.Series(commute, index=o.index))   # fill the few missing commutes with the re-rolled ones from Chapter 2
o["worry_score"] = o.Daily_Commute_km - 5 * o.Charging_Stations_Near_Home - 5 * o.Charging_Stations_Near_Work - 150 * (o.Home_Charging_Possible == "Yes")
fig, ax = plt.subplots(figsize=(13, 4.5))
for lvl, col in zip(["Low", "Medium", "High"], ["#66c2a5", "#ffd92f", "#fc8d62"]): ax.hist(o.worry_score[o.Range_Anxiety_Level == lvl], bins=60, alpha=0.75, color=col, label=lvl, edgecolor="white")
for cut in (-25, 75): ax.axvline(cut, color="black", ls="--", lw=2)
ax.set_xlabel("worry score (without the random wobble)"); ax.set_ylabel("people"); ax.set_title("Range anxiety = worry score + wobble, cut at −25 and +75", fontweight="bold"); ax.legend(title="Range_Anxiety_Level"); plt.show()
bins = pd.cut(o.worry_score, [-400, -150, -100, -60, -40, -25, -10, 10, 30, 50, 75, 100, 150, 300])
share = pd.crosstab(bins, o.Range_Anxiety_Level, normalize="index")[["Low", "Medium", "High"]]
share.style.format("{:.0%}").background_gradient(cmap="YlOrRd", axis=None)

Far left (people with home charging and many chargers) → almost everyone is **Low**. Around the −25 line the colours mix, because the random wobble can push a person over the line either way. Far right → **High**. The dashed lines are exactly where the program cut the score. 📏

## Chapter 5 — The secret buying recipe 🧮🚗

And now the column we care about most. `Will_Buy_EV` is also **computed from a score plus a wobble**:

> **buy score = 1.2 × (income ÷ 100,000) + 0.6 × concern + 2 × subsidy − 1 × (medium anxiety) − 3 × (high anxiety) + random wobble**

* more income → more likely to buy (1.2 points per $100,000),
* caring about the planet → more likely (0.6 per level, so 1 → 5 is worth 2.4 points),
* a government subsidy → a big +2,
* medium worry → −1, high worry → −3,
* the wobble is a bell nudge with spread 1.

If the final score is **above 5.5**, the person buys. Notice which columns are **not** in the recipe: age, gender, city, cars, car type, commute, chargers. They only matter *indirectly* (a big city has fewer home plugs → more worry → fewer buyers).

Let's compute the score without the wobble and check the buying rate along it.

In [ ]:
o["buy_score"] = 1.2 * o.Annual_Income_USD.fillna(pd.Series(income, index=o.index)) / 1e5 + 0.6 * o.Environmental_Concern_Level.fillna(3) + 2.0 * (o.Subsidy_Available == "Yes") - 1.0 * (o.Range_Anxiety_Level == "Medium") - 3.0 * (o.Range_Anxiety_Level == "High")
o["buyer"] = (o.Will_Buy_EV == "Yes").astype(int)
bins = pd.cut(o.buy_score, np.arange(0.5, 9.6, 0.5)); rate = o.groupby(bins, observed=True).buyer.agg(["mean", "size"])
from scipy.stats import norm
fig, ax = plt.subplots(figsize=(12, 5))
centers = [b.mid for b in rate.index]; ax.bar(centers, rate["mean"], width=0.45, color="#8da0cb", edgecolor="white", label="real buying rate in each score bin")
xx = np.linspace(0.5, 9.5, 200); ax.plot(xx, norm.cdf(xx - 5.5), color="red", lw=3, label="what the recipe predicts: chance the wobble pushes the score above 5.5")
ax.axvline(5.5, color="black", ls="--", lw=2); ax.text(5.6, 0.05, "5.5 = the finish line", fontsize=11)
ax.set_xlabel("buy score (without the wobble)"); ax.set_ylabel("share of buyers"); ax.set_ylim(0, 1.02); ax.set_title("Who buys an EV? The recipe explains it almost perfectly", fontweight="bold"); ax.legend(loc="upper left"); plt.show()

The red curve (our recipe) runs right through the blue bars (the real data). 🎯 Below a score of 3 nobody buys; above 8 everybody buys; in between it is a matter of the wobble.

### Can a model ever be perfect here?

No — and that is important for the competition. Two people with **exactly the same facts** can get different answers, because of the random wobble. The best any model can do on the original data is to know the recipe. Let's measure that best-possible score with the competition's metric, **ROC AUC** (1.0 = perfect, 0.5 = coin flip).

In [ ]:
from sklearn.metrics import roc_auc_score
print(f"AUC of the recipe on the original 10,000 people: {roc_auc_score(o.buyer, o.buy_score):.4f}")
print(f"AUC of income alone: {roc_auc_score(o.buyer, o.Annual_Income_USD.fillna(pd.Series(income, index=o.index))):.4f}")
print(f"AUC of environmental concern alone: {roc_auc_score(o.buyer, o.Environmental_Concern_Level.fillna(3)):.4f}")

About **0.90** is the ceiling on the original data — the rest is dice. Any model claiming 0.99 on these 10,000 people would be cheating (or memorising).

## Chapter 6 — From 10,000 to 955,000 people 🧬

The competition organisers took this small dataset and produced a much bigger one: **668,665 training rows and 286,571 test rows**. The new people look very much like the originals — same columns, same kinds of values, the same share of buyers — but there are many more of them. Let's compare a few shares.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col, order in zip(axes[:2], ["City_Type", "Range_Anxiety_Level"], [["Urban", "Suburban", "Rural"], ["Low", "Medium", "High"]]):
    share = pd.DataFrame({"original": orig[col].value_counts(normalize=True), "train": train[col].value_counts(normalize=True)}).reindex(order)
    share.plot(kind="bar", ax=ax, edgecolor="white", color=["#fc8d62", "#8da0cb"]); ax.set_title(col.replace("_", " ") + " shares", fontweight="bold"); ax.tick_params(axis="x", rotation=0)
    for p in ax.patches: ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.005, f"{p.get_height():.0%}", ha="center", fontsize=9)
rates = pd.Series({"original": (orig.Will_Buy_EV == "Yes").mean(), "train": (train.Will_Buy_EV == "Yes").mean()})
axes[2].bar(rates.index, rates.values, color=["#fc8d62", "#8da0cb"], edgecolor="white"); axes[2].set_ylim(0, 0.25); axes[2].set_title("Share of EV buyers", fontweight="bold")
for i, v in enumerate(rates.values): axes[2].text(i, v + 0.005, f"{v:.1%}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

Very similar pictures, so what we learned about the recipe should help on the big dataset too. Let's test that idea.

## Chapter 7 — Two starter models 🤖

### Model 1: the recipe, no training at all

We simply compute the buy score for every training row and score it with AUC. Zero parameters learned!

In [ ]:
def buy_score(df):
    return (1.2 * df.Annual_Income_USD / 1e5 + 0.6 * df.Environmental_Concern_Level + 2.0 * (df.Subsidy_Available == "Yes")
            - 1.0 * (df.Range_Anxiety_Level == "Medium") - 3.0 * (df.Range_Anxiety_Level == "High"))
y = (train.Will_Buy_EV == "Yes").astype(int).values
print(f"Recipe model, AUC on the competition train set: {roc_auc_score(y, buy_score(train)):.4f}   (no training needed!)")

Around **0.938** — much higher than the 0.90 ceiling of the original data! That means the big dataset is a bit *less random* than the original: the recipe explains more of it. Interesting... and it also means there is room for a model that learns extra patterns from the 669,000 rows.

### Model 2: a gradient-boosted tree model (XGBoost)

Tree models are the workhorses of tabular competitions. We do an honest **5-fold cross-validation**: split the training data into 5 parts, train on 4, predict the 5th, and repeat, so every row gets a prediction from a model that never saw it. The test predictions are the average of the 5 models.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
features = [c for c in train.columns if c not in ("id", "Will_Buy_EV")]
X = train[features].copy(); X_test = test[features].copy()
for c in X.select_dtypes("object").columns:                       # turn words into category codes, consistently for train and test
    cats = pd.Categorical(pd.concat([X[c], X_test[c]])).categories
    X[c] = pd.Categorical(X[c], categories=cats).codes; X_test[c] = pd.Categorical(X_test[c], categories=cats).codes
try:
    import cupy; device = "cuda"; print("GPU found — using it")
except Exception:
    device = "cpu"; print("no GPU — using the CPU (this takes a few minutes)")
params = dict(n_estimators=2000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method="hist", device=device, eval_metric="auc", early_stopping_rounds=100)
oof = np.zeros(len(y)); pred_test = np.zeros(len(X_test)); importances = np.zeros(len(features))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = xgb.XGBClassifier(**params)
    model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=False)
    oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]; pred_test += model.predict_proba(X_test)[:, 1] / 5; importances += model.feature_importances_ / 5
    print(f"fold {fold}: AUC {roc_auc_score(y[va_idx], oof[va_idx]):.4f}  (best round {model.best_iteration})")
print(f"\nXGBoost 5-fold cross-validation AUC: {roc_auc_score(y, oof):.4f}")

In [ ]:
imp = pd.Series(importances, index=features).sort_values()
fig, ax = plt.subplots(figsize=(9, 5.5)); imp.plot(kind="barh", ax=ax, color="#66c2a5", edgecolor="white")
ax.set_title("Which facts does XGBoost use most? (the recipe ingredients are at the top)", fontweight="bold"); ax.set_xlabel("importance"); plt.show()

The tree model learned the recipe on its own: income, subsidy, concern and anxiety sit at the top, exactly the ingredients of Chapter 5. 🥇

### Make a submission file 📝

The competition wants one number per test row: the probability that the person buys. We save the XGBoost predictions in the required format. (Upload it on the competition page when you are ready.)

In [ ]:
sub = pd.read_csv(SUB); sub["Will_Buy_EV"] = pred_test
sub.to_csv("submission.csv", index=False); sub.head()

## The end — and your turn! 🏁

What we found today:

* The original 10,000 people were **invented by rolling dice**: a 45-sided die for age, loaded dice for the word columns, bell curves with floors for income and commute — and the computer's lucky seed was **101**.
* **Cities decide chargers and home plugs**, chargers and plugs decide **worry**, and worry plus income, concern and subsidy decide **who buys an EV**, always with a little random wobble.
* On the original data no model can beat about **0.90 AUC** because of that wobble; on the competition data the recipe alone already reaches about **0.938**, and a plain XGBoost reaches about **0.942**.

Ideas to try next (all good practice, none of them secret):

1. Add the two scores from Chapters 4 and 5 as extra columns for XGBoost — do they help the trees?
2. Try other model families (LightGBM, CatBoost, a small neural network) and average their predictions.
3. Tune the tree settings with your 5-fold cross-validation, never on the test set.
4. Look at where the model is most wrong — the people near the "finish line" of the buy score are the hardest.

Thanks for playing detective. Happy modelling! 🚗⚡